# Variational Quantum Regressor: Method 3

This notebook implements a third variational quantum regressor for shallow-water bathymetry. It predicts depth from six engineered spectral features.

This architecture is different from the other project methods: it uses PennyLane's `AngleEmbedding` with Y-axis rotations, `StronglyEntanglingLayers`, target standardization, and a trainable affine classical readout. The target is standardized during optimization and converted back to metres for evaluation.

In [ ]:
%pip install pennylane pennylane-lightning pandas scikit-learn matplotlib

import pennylane as qml
from pennylane import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Load and Prepare the Shallow-Water Dataset

In [ ]:
file_path = "../Dataset/final_sdb_dataset_clean.csv"
df = pd.read_csv(file_path)

features = [
    "blue_band",
    "green_band",
    "log_blue",
    "log_green",
    "bg_ratio",
    "stumpf_ratio"
]
target = "depth"

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=features + [target]).copy()
df = df[(df[target] >= 0) & (df[target] <= 30)].copy()

X = df[features].to_numpy(dtype=float)
y = df[target].to_numpy(dtype=float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

feature_scaler = StandardScaler()
X_train_scaled = feature_scaler.fit_transform(X_train)
X_test_scaled = feature_scaler.transform(X_test)

target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

print(f"Dataset after cleaning: {len(df)} rows")
print(f"Training shape: {X_train_scaled.shape}")
print(f"Testing shape: {X_test_scaled.shape}")

## Method 3 Circuit Architecture

Each standardized feature is encoded with a Y-axis rotation. The trainable circuit uses PennyLane's `StronglyEntanglingLayers`, which applies trainable single-qubit rotations and a built-in entanglement pattern. A trainable scale and bias convert the bounded quantum expectation into standardized depth.

In [ ]:
n_qubits = len(features)
n_layers = 4
dev = qml.device("lightning.qubit", wires=n_qubits)

weight_shape = qml.StronglyEntanglingLayers.shape(
    n_layers=n_layers, n_wires=n_qubits
)

@qml.qnode(dev, interface="autograd")
def vqr_circuit(encoded_features, weights):
    qml.AngleEmbedding(encoded_features, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

print(f"Qubits: {n_qubits}")
print(f"Ansatz layers: {n_layers}")
print(f"Trainable circuit weight shape: {weight_shape}")
print(f"Circuit parameters: {np.prod(weight_shape)} plus scale and bias")

In [ ]:
weights = 0.05 * np.random.randn(*weight_shape, requires_grad=True)
readout_scale = np.array(1.0, requires_grad=True)
readout_bias = np.array(0.0, requires_grad=True)

def predict_scaled(X_batch, circuit_weights, scale, bias):
    quantum_outputs = np.array([
        vqr_circuit(sample, circuit_weights) for sample in X_batch
    ])
    return scale * quantum_outputs + bias

def cost_function(circuit_weights, scale, bias, X_batch, y_batch):
    predictions = predict_scaled(X_batch, circuit_weights, scale, bias)
    return np.mean((predictions - y_batch) ** 2)

## Train the Variational Regressor

The optimizer minimizes mean squared error on standardized training depths. The recorded training loss can be inspected for optimization behaviour. No validation set is used in the original project protocol, so this notebook reports training loss and final held-out test metrics separately.

In [ ]:
optimizer = qml.AdamOptimizer(stepsize=0.05)
epochs = 50
loss_history = []

for epoch in range(epochs):
    (weights, readout_scale, readout_bias), loss = optimizer.step_and_cost(
        cost_function,
        weights,
        readout_scale,
        readout_bias,
        X_batch=X_train_scaled,
        y_batch=y_train_scaled
    )
    loss_history.append(float(loss))
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:02d}/{epochs} | Standardized training MSE: {loss:.6f}")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(1, epochs + 1), loss_history, color="#176B87", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Standardized training MSE")
plt.title("VQR Method 3 Training Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Test Evaluation in Metres

In [ ]:
test_predictions_scaled = predict_scaled(
    X_test_scaled, weights, readout_scale, readout_bias
)
test_predictions = target_scaler.inverse_transform(
    np.asarray(test_predictions_scaled).reshape(-1, 1)
).ravel()

mae = mean_absolute_error(y_test, test_predictions)
rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
r2 = r2_score(y_test, test_predictions)

print("VQR Method 3 results")
print(f"MAE:  {mae:.4f} m")
print(f"RMSE: {rmse:.4f} m")
print(f"R2:   {r2:.4f}")

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, test_predictions, alpha=0.45, color="#D97745")
plt.plot([0, 30], [0, 30], "k--", label="Ideal prediction")
plt.xlabel("Actual depth (m)")
plt.ylabel("Predicted depth (m)")
plt.title(f"VQR Method 3: Actual vs Predicted\nR2 = {r2:.3f}")
plt.xlim(0, 30)
plt.ylim(0, 30)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Comparison Note

Method 3 is intended to be compared with the archived results from `VQR_Method1.ipynb` and `VQR_Method2.ipynb` using the same MAE, RMSE, and R-squared metrics. Its target standardization and affine readout address the output-range mismatch present in the earlier quantum methods, but the result still requires empirical validation.